In [ ]:
# Импортируем необходимые библиотеки
import requests as req  # для выполнения HTTP-запросов
import pandas as pd  # для обработки данных
from datetime import datetime, timedelta  # для работы с датами
import os


# Устанавливаем URL для API
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Параметры запроса
params = {
    'format': 'geojson', # формат выгрузки
    'starttime': datetime.now() - timedelta(days=130), # Дата с: -130 дней от текущей даты(можно подставить любую)
    'endtime': datetime.now(), # Дата по: текущая дата
    'eventtype': 'earthquake', # Тип события
    'minmagnitude': 3.5, # Магнитуда
    'limit': 100, # Кол-во событий
    'orderby': "time" # Сортировка по дате
    }
records = []
# Выполнение запроса
response = req.get(url, params=params)
data = response.json()
# Парсинг файла json для последующей записи в формате csv
for feature in data['features']:
    record = {
            'event_id': feature['id'],
            'magnitude': feature['properties']['mag'],
            'place': feature['properties']['place'],
            'time': datetime.fromtimestamp(feature['properties']['time']/1000).strftime('%Y-%m-%d %H:%M:%S'),
            'load_dt': datetime.now().strftime('%Y-%m-%d %H:%M:%S') # Добавление даты выгрузки
            }
    records.append(record)
# Создаем DataFrame
df = pd.DataFrame(records)

# Создаем папку temp(если нет) и сохраняем данные
os.makedirs('temp', exist_ok=True)

filename = f"temp/earthquakes_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

df.to_csv(filename, index=False, encoding='utf-8')